In [0]:
df_customers = spark.table("bronze_catalog.bronze_stage_schema.customers")
 
# Display the data
display(df_customers)

In [0]:
df= spark.table("bronze_catalog.bronze_stage_schema.invoices")
 
# Display the data
display(df)

In [0]:
df_customers = spark.table("bronze_catalog.bronze_stage_schema.regions")
 
# Display the data
display(df_customers)

In [0]:
%sql
select * from bronze_catalog.bronze_stage_schema.invoices where customer_id ='C0002';

In [0]:
from pyspark.sql.functions import date_format, col, initcap, upper

# Read customers table
df_customers = spark.table("bronze_catalog.bronze_stage_schema.customers")

# Apply all transformations:
# 1. Format signup_date to yyyy-MM-dd
# 2. Convert customer_type to camel case (Title Case)
# 3. Convert country to uppercase
df_customers_transformed = df_customers.withColumn(
    "signup_date", 
    date_format(col("signup_date"), "yyyy-MM-dd")
).withColumn(
    "customer_type",
    initcap(col("customer_type"))
).withColumn(
    "country",
    upper(col("country"))
)

# Display the transformed data
display(df_customers_transformed)

In [0]:
from pyspark.sql.functions import when, col

# Replace U.S.A with USA in the country column
df_customers_final = df_customers_transformed.withColumn(
    "country",
    when(col("country") == "U.S.A", "USA").otherwise(col("country"))
)

# Display the final transformed data
display(df_customers_final)

In [0]:
from pyspark.sql.functions import col, row_number, coalesce, upper
from pyspark.sql.window import Window

# Read invoices and regions tables
df_invoices = spark.table("bronze_catalog.bronze_stage_schema.invoices")
df_regions = spark.table("bronze_catalog.bronze_stage_schema.regions")

# Step 1: Get the latest invoice for each customer
window_spec = Window.partitionBy("customer_id").orderBy(col("invoice_date").desc())
df_latest_invoice = df_invoices.withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .select("customer_id", col("region").alias("invoice_region"))

# Step 2: Join with regions table to get country from region
# Match using region_name (full name) instead of region_code (single letter)
df_regions_cleaned = df_regions.withColumn("region_name_upper", upper(col("region_name")))
df_invoice_country = df_latest_invoice.withColumn("invoice_region_upper", upper(col("invoice_region"))).join(
    df_regions_cleaned,
    col("invoice_region_upper") == col("region_name_upper"),
    "left"
).select("customer_id", col("country").alias("invoice_country"))

# Step 3: Join with customers and fill NULL country values
df_customers_with_country = df_customers_final.join(
    df_invoice_country,
    "customer_id",
    "left"
).withColumn(
    "country",
    coalesce(col("country"), upper(col("invoice_country")))
).drop("invoice_country")

# Display the final result
display(df_customers_with_country)

# Show count of NULL countries before and after
df_customers_with_country.filter(col("customer_id").isin(["C0002", "C0008", "C0016", "C0021"])).show(truncate=False)

In [0]:
# Save the final transformed customers data to silver catalog
df_customers_with_country.write.mode("overwrite").saveAsTable("silver_catalog.silver_schema.customers")

# Verify the save
df_verify = spark.table("silver_catalog.silver_schema.customers")
df_verify.show(5, truncate=False)

In [0]:
# Take the final cleaned customers DataFrame from previous cell
# and write to silver_catalog.silver_schema.customers

df_customers_with_country.write.mode("overwrite").saveAsTable("silver_catalog.silver_schema.customers")

df_silver_customers = spark.table("silver_catalog.silver_schema.customers")

display(df_silver_customers.limit(10))

In [0]:
df_customers_with_country.printSchema()